In [ ]:
# ============================================================
# BLOCK 4: FEATURE ENGINEERING — ADDING NEW FEATURES
# Project: Forecasting Household Deposit Volume in Russia
# Author: Nadezhda Silkina
# Date: 2026
# ============================================================

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Libraries loaded")

# ============================================================
# 2. LOAD DATA AND PREPARE BASELINE MODEL
# ============================================================

url = 'https://raw.githubusercontent.com/HopeSilkina/deposits_forecast_project/main/data/processed_deposits_data.xlsx'
df = pd.read_excel(url, sheet_name='data')
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

# Create baseline features (DEPOS lags)
df['DEPOS_log'] = np.log(df['DEPOS'])
for lag in [1, 3, 6, 12]:
    df[f'DEPOS_lag_{lag}'] = df['DEPOS'].shift(lag)

# Prepare X and y
X_base = df.drop(['DEPOS', 'DEPOS_log'], axis=1).dropna()
y = df.loc[X_base.index, 'DEPOS']

# Split into train/test
train_size = len(X_base) - 12
X_train_base, X_test_base = X_base.iloc[:train_size], X_base.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

# Scaling for baseline model
scaler_base = StandardScaler()
X_train_scaled_base = scaler_base.fit_transform(X_train_base)
X_test_scaled_base = scaler_base.transform(X_test_base)

# Baseline Ridge model
ridge_base = Ridge(alpha=1.0)
ridge_base.fit(X_train_scaled_base, y_train)
y_pred_base = ridge_base.predict(X_test_scaled_base)

r2_base = r2_score(y_test, y_pred_base)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_base))
mae_base = mean_absolute_error(y_test, y_pred_base)

print("="*60)
print("BASELINE MODEL (RIDGE, ALPHA=1.0)")
print("="*60)
print(f"R² = {r2_base:.4f}")
print(f"RMSE = {rmse_base:.2f} billion RUB")
print(f"MAE = {mae_base:.2f} billion RUB")

# ============================================================
# 3. ADDING NEW FEATURES
# ============================================================

print("\n" + "="*60)
print("ADDING NEW FEATURES")
print("="*60)

# Copy data for new features
df_feat = df.copy()

# 3.1. Seasonal dummy variables
print("\n🔍 Adding seasonal dummy variables...")
df_feat['Month'] = df_feat.index.month
month_dummies = pd.get_dummies(df_feat['Month'], prefix='month', drop_first=True)
df_feat = pd.concat([df_feat, month_dummies], axis=1)
print(f"   Added 11 seasonal variables")

# 3.2. Dummy variable post_2022
print("\n🔍 Adding dummy variable post_2022...")
df_feat['post_2022'] = (df_feat.index >= '2023-01-01').astype(int)
print(f"   post_2022 = 1 for {df_feat['post_2022'].sum()} records (since 2023)")

# 3.3. Dummy variable covid
print("\n🔍 Adding dummy variable covid...")
df_feat['covid'] = ((df_feat.index >= '2020-03-01') & (df_feat.index <= '2022-01-01')).astype(int)
print(f"   covid = 1 for {df_feat['covid'].sum()} records")

# 3.4. Dummy variable regime_cred1 (based on DEPOS)
print("\n🔍 Adding dummy variable regime_cred1...")
threshold_depos = 32000
df_feat['regime_cred1'] = (df_feat['DEPOS'] > threshold_depos).astype(int)
print(f"   regime_cred1 = 1 for {df_feat['regime_cred1'].sum()} records (DEPOS > {threshold_depos})")

# 3.5. Dummy variable anomaly_wage (with adaptive threshold)
print("\n🔍 Adding dummy variable anomaly_wage with adaptive threshold...")

# Function to test the impact of anomaly_wage with a given threshold
def test_anomaly_threshold(thresh, df_feat, X_train_base, X_test_base, y_train, y_test, r2_base):
    # Calculate anomalies
    model_wage = LinearRegression()
    model_wage.fit(df_feat[['WAGE']].values, df_feat['DEPOS'].values)
    df_feat['residual_wage'] = df_feat['DEPOS'] - model_wage.predict(df_feat[['WAGE']].values)
    threshold_anomaly = thresh * df_feat['residual_wage'].std()
    df_feat['anomaly_wage'] = (df_feat['residual_wage'] < threshold_anomaly).astype(int)

    # Create X with new feature
    X_train_temp = X_train_base.copy()
    X_test_temp = X_test_base.copy()
    X_train_temp['anomaly_wage'] = df_feat.loc[X_train_temp.index, 'anomaly_wage']
    X_test_temp['anomaly_wage'] = df_feat.loc[X_test_temp.index, 'anomaly_wage']

    # Scale (with new feature)
    scaler_temp = StandardScaler()
    X_train_scaled_temp = scaler_temp.fit_transform(X_train_temp)
    X_test_scaled_temp = scaler_temp.transform(X_test_temp)

    # Train model
    ridge_temp = Ridge(alpha=1.0)
    ridge_temp.fit(X_train_scaled_temp, y_train)
    y_pred_temp = ridge_temp.predict(X_test_scaled_temp)
    r2_temp = r2_score(y_test, y_pred_temp)

    return r2_temp, df_feat['anomaly_wage'].sum()

# Try different thresholds
thresholds = [-1.0, -1.2, -1.5, -2.0]
best_r2_anomaly = r2_base
best_threshold = None
best_anomaly_count = 0

for thresh in thresholds:
    r2_temp, count = test_anomaly_threshold(thresh, df_feat, X_train_base, X_test_base, y_train, y_test, r2_base)
    print(f"   Threshold {thresh}σ: anomalies = {count}, R² = {r2_temp:.4f}")

    if r2_temp > best_r2_anomaly:
        best_r2_anomaly = r2_temp
        best_threshold = thresh
        best_anomaly_count = count

# Save the best version of anomaly_wage
if best_threshold is not None:
    model_wage = LinearRegression()
    model_wage.fit(df_feat[['WAGE']].values, df_feat['DEPOS'].values)
    df_feat['residual_wage'] = df_feat['DEPOS'] - model_wage.predict(df_feat[['WAGE']].values)
    threshold_anomaly_best = best_threshold * df_feat['residual_wage'].std()
    df_feat['anomaly_wage'] = (df_feat['residual_wage'] < threshold_anomaly_best).astype(int)
    print(f"\n   ✅ Best threshold: {best_threshold}σ (anomalies: {best_anomaly_count})")
    print(f"   ✅ R² improvement: {best_r2_anomaly - r2_base:.4f}")

# 3.6. Lags for WAGE, CPI, USDind
print("\n🔍 Adding lags for WAGE, CPI, USDind...")
for col in ['WAGE', 'CPI', 'USDind']:
    for lag in [1, 3, 6]:
        df_feat[f'{col}_lag_{lag}'] = df_feat[col].shift(lag)
print(f"   Added 9 lag features (3 variables × 3 lags)")

# 3.7. Interaction UNEM × DEP1
print("\n🔍 Adding interaction UNEM × DEP1...")
df_feat['UNEM_DEP1'] = df_feat['UNEM'] * df_feat['DEP1']
print("   Added interaction UNEM × DEP1")

# ============================================================
# 4. DATA PREPARATION FOR MODEL WITH NEW FEATURES
# ============================================================

print("\n" + "="*60)
print("DATA PREPARATION FOR MODEL WITH NEW FEATURES")
print("="*60)

# Remove auxiliary columns
X_new = df_feat.drop(['DEPOS', 'DEPOS_log', 'Month', 'residual_wage'], axis=1).dropna()

print(f"📊 Number of features in baseline model: {X_base.shape[1]}")
print(f"📊 Number of features in new model: {X_new.shape[1]}")
print(f"📊 Features added: {X_new.shape[1] - X_base.shape[1]}")

# Split into train/test
X_train_new, X_test_new = X_new.iloc[:train_size], X_new.iloc[train_size:]

# Scaling
scaler_new = StandardScaler()
X_train_scaled_new = scaler_new.fit_transform(X_train_new)
X_test_scaled_new = scaler_new.transform(X_test_new)

# ============================================================
# 5. TRAIN MODEL WITH NEW FEATURES
# ============================================================

print("\n" + "="*60)
print("TRAINING MODEL WITH NEW FEATURES")
print("="*60)

ridge_new = Ridge(alpha=1.0)
ridge_new.fit(X_train_scaled_new, y_train)
y_pred_new = ridge_new.predict(X_test_scaled_new)

r2_new = r2_score(y_test, y_pred_new)
rmse_new = np.sqrt(mean_squared_error(y_test, y_pred_new))
mae_new = mean_absolute_error(y_test, y_pred_new)

print(f"\n📊 Results of model with new features:")
print(f"   R² = {r2_new:.4f}")
print(f"   RMSE = {rmse_new:.2f} billion RUB")
print(f"   MAE = {mae_new:.2f} billion RUB")

# ============================================================
# 6. MODEL COMPARISON
# ============================================================

print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)

comparison = pd.DataFrame({
    'Model': ['Baseline (Ridge)', 'With new features'],
    'R²': [r2_base, r2_new],
    'RMSE': [rmse_base, rmse_new],
    'MAE': [mae_base, mae_new],
    'Features': [X_base.shape[1], X_new.shape[1]]
})

print(comparison.to_string(index=False))

improvement = r2_new - r2_base
print(f"\n📊 R² improvement: {improvement:.4f}")

if improvement > 0.01:
    print("   ✅ New features SIGNIFICANTLY improve the model")
elif improvement > 0:
    print("   ✅ New features SLIGHTLY improve the model")
else:
    print("   ℹ️ New features DO NOT improve the model")

# ============================================================
# 7. ANALYSIS OF EACH NEW FEATURE
# ============================================================

print("\n" + "="*60)
print("ANALYSIS OF EACH NEW FEATURE")
print("="*60)

def test_feature(X_train_base, X_test_base, feature_name, y_train, y_test, r2_base):
    X_train_test = X_train_base.copy()
    X_test_test = X_test_base.copy()
    X_train_test[feature_name] = X_train_new[feature_name]
    X_test_test[feature_name] = X_test_new[feature_name]

    scaler_test = StandardScaler()
    X_train_scaled_test = scaler_test.fit_transform(X_train_test)
    X_test_scaled_test = scaler_test.transform(X_test_test)

    ridge_test = Ridge(alpha=1.0)
    ridge_test.fit(X_train_scaled_test, y_train)
    y_pred_test = ridge_test.predict(X_test_scaled_test)
    r2_test = r2_score(y_test, y_pred_test)
    improvement = r2_test - r2_base
    return improvement

new_features = [
    'month_2', 'month_3', 'month_4', 'month_5', 'month_6',
    'month_7', 'month_8', 'month_9', 'month_10', 'month_11', 'month_12',
    'post_2022', 'covid', 'regime_cred1', 'anomaly_wage',
    'WAGE_lag_1', 'WAGE_lag_3', 'WAGE_lag_6',
    'CPI_lag_1', 'CPI_lag_3', 'CPI_lag_6',
    'USDind_lag_1', 'USDind_lag_3', 'USDind_lag_6',
    'UNEM_DEP1'
]

print("\n🔍 Impact of each new feature on R²:")
results = []
for feature in new_features:
    imp = test_feature(X_train_base, X_test_base, feature, y_train, y_test, r2_base)
    results.append({'Feature': feature, 'R² Improvement': imp})
    status = '✅' if imp > 0.001 else 'ℹ️'
    print(f"   {status} {feature}: {imp:.4f}")

results_df = pd.DataFrame(results).sort_values('R² Improvement', ascending=False)
print("\n📊 Top-5 features by improvement:")
print(results_df.head(5).to_string(index=False))

# ============================================================
# 8. FINAL SUMMARY
# ============================================================

print("\n" + "="*60)
print("📌 FINAL SUMMARY FOR BLOCK 4")
print("="*60)

best_r2 = max(r2_base, r2_new)
best_model = 'Baseline' if r2_base >= r2_new else 'With new features'

print(f"\n🏆 Best model: {best_model} (R² = {best_r2:.4f})")

if improvement > 0.01:
    print("\n✅ Adding new features SIGNIFICANTLY improves the model")
elif improvement > 0:
    print("\n✅ Adding new features SLIGHTLY improves the model")
else:
    print("\nℹ️ Adding new features DO NOT improve the model")

print("\n📌 KEY FINDINGS:")
print("   1. Features with the greatest improvement: " +
      ", ".join(results_df.head(3)['Feature'].tolist()))
print("   2. Seasonal variables " +
      ("improve" if any('month' in r['Feature'] and r['R² Improvement'] > 0.001 for r in results) else "do not improve") + " the model")
print("   3. Dummy variable post_2022 " +
      ("improves" if results_df[results_df['Feature'] == 'post_2022']['R² Improvement'].values[0] > 0.001 else "does not improve") + " the model")
print("   4. Lags for WAGE, CPI, USDind " +
      ("improve" if any('_lag_' in r['Feature'] and r['R² Improvement'] > 0.001 for r in results) else "do not improve") + " the model")

# ============================================================
# 9. VISUALIZATION OF DUMMY VARIABLE IMPACT
# ============================================================

print("\n" + "="*60)
print("9. VISUALIZATION OF DUMMY VARIABLE IMPACT")
print("="*60)

# Get predictions from baseline model and model with new features
# (on the full dataset to see all points)
X_full_base = X_base.copy()
X_full_new = X_new.copy()

# Scale full data
scaler_full_base = StandardScaler()
X_full_scaled_base = scaler_full_base.fit_transform(X_full_base)

scaler_full_new = StandardScaler()
X_full_scaled_new = scaler_full_new.fit_transform(X_full_new)

# Predictions
y_pred_full_base = ridge_base.predict(X_full_scaled_base)
y_pred_full_new = ridge_new.predict(X_full_scaled_new)

# Create DataFrame for visualization
df_plot = df_feat.loc[X_full_new.index].copy()
df_plot['DEPOS_actual'] = y.loc[X_full_new.index]
df_plot['DEPOS_pred_base'] = y_pred_full_base
df_plot['DEPOS_pred_new'] = y_pred_full_new
df_plot['DEPOS_error_new'] = df_plot['DEPOS_actual'] - df_plot['DEPOS_pred_new']

# 9.1. Plot 1: DEPOS vs WAGE with adaptive anomaly threshold
print("\n🔍 Plot 1: DEPOS vs WAGE (anomalies with adaptive threshold)")

# Use the best threshold from adaptive selection
if best_threshold is not None:
    threshold_anomaly_best = best_threshold * df_feat['residual_wage'].std()
    df_plot['anomaly_wage_best'] = (df_feat.loc[df_plot.index, 'residual_wage'] < threshold_anomaly_best).astype(int)
    print(f"   Threshold: {best_threshold}σ, anomalies: {df_plot['anomaly_wage_best'].sum()}")
else:
    threshold_anomaly_best = -1.5 * df_feat['residual_wage'].std()
    df_plot['anomaly_wage_best'] = (df_feat.loc[df_plot.index, 'residual_wage'] < threshold_anomaly_best).astype(int)
    print(f"   Using threshold -1.5σ, anomalies: {df_plot['anomaly_wage_best'].sum()}")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left plot: actual data with anomaly highlighting
ax = axes[0]
scatter = ax.scatter(df_plot['WAGE'], df_plot['DEPOS_actual'],
                     c=df_plot['anomaly_wage_best'], cmap='coolwarm', alpha=0.7, s=30)
ax.set_xlabel('WAGE (RUB)')
ax.set_ylabel('DEPOS (billion RUB)')
ax.set_title(f'Actual data: anomalies (threshold {best_threshold}σ)' if best_threshold else 'Actual data: anomalies')
ax.legend(*scatter.legend_elements(), title="anomaly")
ax.grid(True, alpha=0.3)

# Right plot: predicted values with anomaly highlighting
ax = axes[1]
scatter = ax.scatter(df_plot['WAGE'], df_plot['DEPOS_pred_new'],
                     c=df_plot['anomaly_wage_best'], cmap='coolwarm', alpha=0.7, s=30)
ax.set_xlabel('WAGE (RUB)')
ax.set_ylabel('DEPOS (billion RUB)')
ax.set_title('Model predictions: anomalies highlighted')
ax.legend(*scatter.legend_elements(), title="anomaly")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('04_anomaly_wage.png', dpi=300, bbox_inches='tight')
plt.show()

# 9.2. Plot 2: DEPOS vs UNEM with covid
print("\n🔍 Plot 2: DEPOS vs UNEM (comparing models with and without covid)")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left plot: predictions without covid
ax = axes[0]
scatter = ax.scatter(df_plot['UNEM'], df_plot['DEPOS_pred_base'],
                     c=df_plot['covid'], cmap='coolwarm', alpha=0.7, s=30)
ax.set_xlabel('UNEM (%)')
ax.set_ylabel('DEPOS (billion RUB)')
ax.set_title('Predictions WITHOUT covid')
ax.legend(*scatter.legend_elements(), title="covid")
ax.grid(True, alpha=0.3)

# Right plot: predictions with covid
ax = axes[1]
scatter = ax.scatter(df_plot['UNEM'], df_plot['DEPOS_pred_new'],
                     c=df_plot['covid'], cmap='coolwarm', alpha=0.7, s=30)
ax.set_xlabel('UNEM (%)')
ax.set_ylabel('DEPOS (billion RUB)')
ax.set_title('Predictions WITH covid')
ax.legend(*scatter.legend_elements(), title="covid")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('04_covid_unem.png', dpi=300, bbox_inches='tight')
plt.show()

# 9.3. Plot 3: DEPOS vs CRED1 with regression lines for clusters
print("\n🔍 Plot 3: DEPOS vs CRED1 (regression lines for clusters)")

# Split by DEPOS (correct)
threshold_depos = 32000
mask1 = df_plot['DEPOS_actual'] <= threshold_depos
mask2 = df_plot['DEPOS_actual'] > threshold_depos

print(f"   Cluster 1 (DEPOS <= {threshold_depos}): {mask1.sum()} records")
print(f"   Cluster 2 (DEPOS > {threshold_depos}): {mask2.sum()} records")

# Function to add regression line
def add_regression_line(ax, x, y, color, label):
    if len(x) < 2:
        print(f"   ⚠️ Insufficient data for regression: {label} (n={len(x)})")
        return
    model = LinearRegression()
    model.fit(x.values.reshape(-1, 1), y.values)
    x_range = np.linspace(x.min(), x.max(), 100)
    y_range = model.predict(x_range.reshape(-1, 1))
    ax.plot(x_range, y_range, color=color, linewidth=2, label=label)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left plot: actual data
ax = axes[0]
ax.scatter(df_plot.loc[mask1, 'CRED1'], df_plot.loc[mask1, 'DEPOS_actual'],
           color='blue', alpha=0.6, s=30, label='Cluster 1 (DEPOS ≤ 32000)')
add_regression_line(ax, df_plot.loc[mask1, 'CRED1'], df_plot.loc[mask1, 'DEPOS_actual'],
                    'blue', 'Cluster 1 trend')
ax.scatter(df_plot.loc[mask2, 'CRED1'], df_plot.loc[mask2, 'DEPOS_actual'],
           color='red', alpha=0.6, s=30, label='Cluster 2 (DEPOS > 32000)')
add_regression_line(ax, df_plot.loc[mask2, 'CRED1'], df_plot.loc[mask2, 'DEPOS_actual'],
                    'red', 'Cluster 2 trend')
ax.axhline(y=threshold_depos, color='green', linestyle='--', alpha=0.5,
           label=f'DEPOS threshold = {threshold_depos}')
ax.set_xlabel('CRED1 (%)')
ax.set_ylabel('DEPOS (billion RUB)')
ax.set_title('Actual data: two clusters')
ax.legend()
ax.grid(True, alpha=0.3)

# Right plot: model predictions
ax = axes[1]
ax.scatter(df_plot.loc[mask1, 'CRED1'], df_plot.loc[mask1, 'DEPOS_pred_new'],
           color='blue', alpha=0.6, s=30, label='Cluster 1')
add_regression_line(ax, df_plot.loc[mask1, 'CRED1'], df_plot.loc[mask1, 'DEPOS_pred_new'],
                    'blue', 'Prediction trend')
ax.scatter(df_plot.loc[mask2, 'CRED1'], df_plot.loc[mask2, 'DEPOS_pred_new'],
           color='red', alpha=0.6, s=30, label='Cluster 2')
add_regression_line(ax, df_plot.loc[mask2, 'CRED1'], df_plot.loc[mask2, 'DEPOS_pred_new'],
                    'red', 'Prediction trend')
ax.axhline(y=threshold_depos, color='green', linestyle='--', alpha=0.5,
           label=f'DEPOS threshold = {threshold_depos}')
ax.set_xlabel('CRED1 (%)')
ax.set_ylabel('DEPOS (billion RUB)')
ax.set_title('Model predictions: two clusters')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('04_regime_cred1.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Visualization completed")


# ============================================================
# 10. REMOVING INSIGNIFICANT FEATURES (CORRECTED VERSION)
# ============================================================

print("\n" + "="*60)
print("10. REMOVING INSIGNIFICANT FEATURES")
print("="*60)

import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_breusch_godfrey
from scipy.stats import shapiro

# -------------------------------------------------------------------
# 10.1. Manual feature removal (based on analysis)
# -------------------------------------------------------------------

print("\n" + "-"*60)
print("OPTION 1: Manual feature removal")
print("-"*60)

# List of features to remove (based on analysis from key_insights_4)
features_to_drop = [
    'regime_cred1',      # worsens the model (-0.1148)
    'month_7',           # negative impact (-0.0027)
    'month_8',           # negative impact (-0.0104)
    'CPI_lag_6',         # negative impact (-0.0139)
    'USDind_lag_3',      # negative impact (-0.0185)
    'USDind_lag_1',      # negative impact (-0.0095)
    'WAGE_lag_6',        # negative impact (-0.0005)
    'CPI_lag_3',         # negative impact (-0.0032)
]

print(f"\n🔍 Features to remove ({len(features_to_drop)} pcs.):")
for f in features_to_drop:
    print(f"   - {f}")

# Create X with removed features
X_manual = X_new.drop(columns=features_to_drop, axis=1)

# Split into train/test
X_train_manual, X_test_manual = X_manual.iloc[:train_size], X_manual.iloc[train_size:]

# Scaling
scaler_manual = StandardScaler()
X_train_scaled_manual = scaler_manual.fit_transform(X_train_manual)
X_test_scaled_manual = scaler_manual.transform(X_test_manual)

# Train model
ridge_manual = Ridge(alpha=1.0)
ridge_manual.fit(X_train_scaled_manual, y_train)

# Predictions on BOTH samples
y_train_pred_manual = ridge_manual.predict(X_train_scaled_manual)
y_test_pred_manual = ridge_manual.predict(X_test_scaled_manual)

# Metrics on TEST sample (predictive ability)
r2_test_manual = r2_score(y_test, y_test_pred_manual)
rmse_test_manual = np.sqrt(mean_squared_error(y_test, y_test_pred_manual))
mae_test_manual = mean_absolute_error(y_test, y_test_pred_manual)

# Metrics on TRAIN sample (goodness of fit)
r2_train_manual = r2_score(y_train, y_train_pred_manual)

print(f"\n📊 Results (manual removal):")
print(f"   TRAINING sample:")
print(f"     R²_train = {r2_train_manual:.4f}")
print(f"   TEST sample:")
print(f"     R²_test = {r2_test_manual:.4f}")
print(f"     RMSE = {rmse_test_manual:.2f} billion RUB")
print(f"     MAE = {mae_test_manual:.2f} billion RUB")
print(f"   Features: {X_manual.shape[1]}")

# -------------------------------------------------------------------
# 10.2. Stepwise Selection
# -------------------------------------------------------------------

print("\n" + "-"*60)
print("OPTION 2: Stepwise Selection")
print("-"*60)

from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression

print("\n🔍 Performing stepwise selection (may take several minutes)...")

# Stepwise using LinearRegression
sfs = SequentialFeatureSelector(
    LinearRegression(),
    n_features_to_select='auto',
    direction='forward',
    scoring='r2',
    cv=5,
    n_jobs=-1,
    tol=0.001
)

# Use X_train_base (unscaled data with names)
sfs.fit(X_train_base, y_train)

# Get selected features
selected_features_stepwise = X_train_base.columns[sfs.get_support()].tolist()

print(f"\n📊 Selected {len(selected_features_stepwise)} features:")
print(f"   {selected_features_stepwise}")

# Create X with selected features
X_stepwise = X_new[selected_features_stepwise]

# Split into train/test
X_train_step, X_test_step = X_stepwise.iloc[:train_size], X_stepwise.iloc[train_size:]

# Scaling
scaler_step = StandardScaler()
X_train_scaled_step = scaler_step.fit_transform(X_train_step)
X_test_scaled_step = scaler_step.transform(X_test_step)

# Train model
ridge_step = Ridge(alpha=1.0)
ridge_step.fit(X_train_scaled_step, y_train)

# Predictions on BOTH samples
y_train_pred_step = ridge_step.predict(X_train_scaled_step)
y_test_pred_step = ridge_step.predict(X_test_scaled_step)

# Metrics on TEST sample
r2_test_step = r2_score(y_test, y_test_pred_step)
rmse_test_step = np.sqrt(mean_squared_error(y_test, y_test_pred_step))
mae_test_step = mean_absolute_error(y_test, y_test_pred_step)

# Metrics on TRAIN sample
r2_train_step = r2_score(y_train, y_train_pred_step)

print(f"\n📊 Results (stepwise selection):")
print(f"   TRAINING sample:")
print(f"     R²_train = {r2_train_step:.4f}")
print(f"   TEST sample:")
print(f"     R²_test = {r2_test_step:.4f}")
print(f"     RMSE = {rmse_test_step:.2f} billion RUB")
print(f"     MAE = {mae_test_step:.2f} billion RUB")
print(f"   Features: {X_stepwise.shape[1]}")

# Add predictions for full Ridge model on training sample
y_train_pred_new = ridge_new.predict(X_train_scaled_new)
r2_train_new = r2_score(y_train, y_train_pred_new)

print(f"\n📊 Results of full Ridge model (for reference):")
print(f"   TRAINING sample:")
print(f"     R²_train = {r2_train_new:.4f}")
print(f"   TEST sample:")
print(f"     R²_test = {r2_new:.4f}")
print(f"   Features: {X_new.shape[1]}")

# -------------------------------------------------------------------
# 10.3. EXTENDED MODEL COMPARISON
# -------------------------------------------------------------------

print("\n" + "="*60)
print("EXTENDED MODEL COMPARISON")
print("="*60)

def calculate_aic_bic_corrected(y_true, y_pred, n_params):
    """
    Calculate AIC and BIC on training sample.

    Uses unbiased estimator for variance: sigma2 = RSS / (n - n_params)
    This provides a more correct estimate of log-likelihood.

    Parameters:
    ----------
    y_true : array-like
        Actual values (training sample)
    y_pred : array-like
        Predicted values (on training sample)
    n_params : int
        Number of model parameters (including intercept for Ridge)

    Returns:
    ----------
    aic, bic : float
        Information criterion values
    """
    n = len(y_true)
    residuals = y_true - y_pred
    rss = np.sum(residuals**2)

    # CORRECTION: use unbiased estimator for variance
    sigma2 = rss / (n - n_params)  # unbiased estimator

    # Log-likelihood for normal distribution
    log_likelihood = -0.5 * n * (np.log(2 * np.pi * sigma2) + 1)

    aic = -2 * log_likelihood + 2 * n_params
    bic = -2 * log_likelihood + n_params * np.log(n)

    return aic, bic

def adjusted_r2_train(r2_train, n_train, n_params):
    """
    Calculate adjusted R² on training sample.

    Parameters:
    ----------
    r2_train : float
        R² on training sample
    n_train : int
        Number of observations in training sample
    n_params : int
        Number of model parameters

    Returns:
    ----------
    adj_r2 : float
        Adjusted R²
    """
    if n_train - n_params - 1 <= 0:
        return np.nan
    return 1 - (1 - r2_train) * (n_train - 1) / (n_train - n_params - 1)

n_train = len(y_train)  # 122
n_test = len(y_test)    # 12

# For Ridge, add intercept as a parameter
# Ridge includes intercept if fit_intercept=True (default)
intercept_param = 1

# Collect all models with comparison data
models_data = [
    {
        'name': 'Full Ridge (all features)',
        'y_train_pred': y_train_pred_new,
        'y_test_pred': y_pred_new,
        'n_params': X_new.shape[1] + intercept_param,
        'r2_train': r2_train_new,
        'r2_test': r2_new
    },
    {
        'name': 'Manual removal',
        'y_train_pred': y_train_pred_manual,
        'y_test_pred': y_test_pred_manual,
        'n_params': X_manual.shape[1] + intercept_param,
        'r2_train': r2_train_manual,
        'r2_test': r2_test_manual
    },
    {
        'name': 'Stepwise selection',
        'y_train_pred': y_train_pred_step,
        'y_test_pred': y_test_pred_step,
        'n_params': X_stepwise.shape[1] + intercept_param,
        'r2_train': r2_train_step,
        'r2_test': r2_test_step
    }
]

results_all = []

for model in models_data:
    # Metrics on training sample
    r2_train = model['r2_train']
    adj_r2 = adjusted_r2_train(r2_train, n_train, model['n_params'])
    aic, bic = calculate_aic_bic_corrected(y_train, model['y_train_pred'], model['n_params'])

    # Metrics on test sample (predictive ability)
    r2_test = model['r2_test']
    rmse_test = np.sqrt(mean_squared_error(y_test, model['y_test_pred']))
    mae_test = mean_absolute_error(y_test, model['y_test_pred'])

    # R²_gap: difference between training and test quality
    r2_gap = r2_train - r2_test

    results_all.append({
        'Model': model['name'],
        'Features': model['n_params'] - intercept_param,  # without intercept
        'R²_train': r2_train,
        'R²_adj_train': adj_r2,
        'R²_test': r2_test,
        'R²_gap': r2_gap,
        'RMSE_test': rmse_test,
        'MAE_test': mae_test,
        'AIC': aic,
        'BIC': bic
    })

df_comparison = pd.DataFrame(results_all)

print("\n📊 Model comparison table (metrics on TRAINING sample):")
print("─" * 80)
print(f"{'Model':<30} {'Feat.':<8} {'R²_train':<10} {'R²_adj':<10} {'AIC':<12} {'BIC':<12}")
print("─" * 80)
for _, row in df_comparison.iterrows():
    print(f"{row['Model']:<30} {row['Features']:<8} {row['R²_train']:<10.4f} {row['R²_adj_train']:<10.4f} {row['AIC']:<12.2f} {row['BIC']:<12.2f}")

print("\n📊 Metrics on TEST sample (predictive ability):")
print("─" * 80)
print(f"{'Model':<30} {'R²_test':<10} {'RMSE_test':<12} {'MAE_test':<12} {'R²_gap':<10}")
print("─" * 80)
for _, row in df_comparison.iterrows():
    print(f"{row['Model']:<30} {row['R²_test']:<10.4f} {row['RMSE_test']:<12.2f} {row['MAE_test']:<12.2f} {row['R²_gap']:<10.4f}")

print("\n📌 R²_gap = R²_train - R²_test — difference between training and")
print("   test quality. Large R²_gap indicates overfitting.")

# -------------------------------------------------------------------
# 10.4. RESIDUAL DIAGNOSTICS ON TRAINING SAMPLE
# -------------------------------------------------------------------

print("\n" + "-"*60)
print("RESIDUAL DIAGNOSTICS (on TRAINING sample)")
print("-"*60)

def diagnose_residuals_train(y_train, y_train_pred, model_name):
    """
    Comprehensive residual diagnostics on TRAINING sample.

    Tests performed:
    1. Shapiro-Wilk test - normality of residuals
    2. Breusch-Pagan test - homoscedasticity
    3. Breusch-Godfrey test - autocorrelation (instead of DW)

    IMPORTANT: All tests are performed on the training sample!
    """
    residuals = y_train - y_train_pred
    n = len(residuals)

    # 1. Shapiro-Wilk test (normality)
    if n >= 3 and n <= 5000:  # Shapiro-Wilk works for 3 <= n <= 5000
        shapiro_stat, shapiro_p = shapiro(residuals)
        normality = '✅ Normal' if shapiro_p > 0.05 else '⚠️ Deviation'
    else:
        shapiro_stat, shapiro_p = np.nan, np.nan
        normality = 'N/A (n out of range)'

    # 2. Breusch-Pagan test (homoscedasticity)
    try:
        # Use predicted values as exogenous variable
        exog_bp = sm.add_constant(y_train_pred.reshape(-1, 1))
        bp_stat, bp_p, bp_f, bp_f_p = het_breuschpagan(residuals, exog_bp)
        homoscedasticity = '✅ Homoscedastic' if bp_p > 0.05 else '⚠️ Heteroscedastic'
    except:
        bp_stat, bp_p = np.nan, np.nan
        homoscedasticity = 'Calculation error'

    # 3. Breusch-Godfrey test (autocorrelation) - REPLACEMENT for DW test
    # Advantages over DW:
    # - Works with lagged dependent variables
    # - Allows testing higher-order autocorrelation
    # - More powerful for small samples
    try:
        # Create feature matrix (use constant and predicted values)
        exog_bg = sm.add_constant(y_train_pred.reshape(-1, 1))

        # Test for 1st-order autocorrelation (nlags=1)
        bg_stat, bg_p, bg_f, bg_f_p = acorr_breusch_godfrey(
            sm.OLS(residuals, exog_bg).fit(),
            nlags=1
        )
        autocorrelation = '✅ No autocorrelation' if bg_p > 0.05 else '⚠️ Autocorrelation'

        # Additionally test for 4th-order autocorrelation (seasonal)
        bg_stat_4, bg_p_4, _, _ = acorr_breusch_godfrey(
            sm.OLS(residuals, exog_bg).fit(),
            nlags=4
        )
        autocorrelation_4 = '✅ None' if bg_p_4 > 0.05 else '⚠️ Present (order 4)'
    except:
        bg_stat, bg_p = np.nan, np.nan
        bg_stat_4, bg_p_4 = np.nan, np.nan
        autocorrelation = 'Calculation error'
        autocorrelation_4 = 'Calculation error'

    return {
        'Model': model_name,
        'Shapiro-Wilk stat': shapiro_stat,
        'Shapiro-Wilk p': shapiro_p,
        'Normality': normality,
        'Breusch-Pagan p': bp_p,
        'Homoscedasticity': homoscedasticity,
        'BG test (lag 1) p': bg_p,
        'Autocorrelation (lag 1)': autocorrelation,
        'BG test (lag 4) p': bg_p_4,
        'Autocorrelation (lag 4)': autocorrelation_4
    }

# Perform diagnostics for all models
print("\n🔍 Performing residual diagnostics...")
print("   IMPORTANT: All tests are performed on the TRAINING sample (n=122)")

diagnostic_results = []
for model in models_data:
    result = diagnose_residuals_train(
        y_train,
        model['y_train_pred'],
        model['name']
    )
    diagnostic_results.append(result)

df_diagnostic = pd.DataFrame(diagnostic_results)

print("\n📊 Residual diagnostics results (training sample):")
print("=" * 80)
for _, row in df_diagnostic.iterrows():
    print(f"\n{row['Model']}:")
    print(f"  Normality: {row['Normality']} (p = {row['Shapiro-Wilk p']:.4f})")
    print(f"  Homoscedasticity: {row['Homoscedasticity']} (p = {row['Breusch-Pagan p']:.4f})")
    print(f"  Autocorrelation (lag 1): {row['Autocorrelation (lag 1)']} (p = {row['BG test (lag 1) p']:.4f})")
    print(f"  Autocorrelation (lag 4): {row['Autocorrelation (lag 4)']} (p = {row['BG test (lag 4) p']:.4f})")

# -------------------------------------------------------------------
# 10.5. RESIDUAL PLOTS FOR BEST MODEL (on training sample)
# -------------------------------------------------------------------

print("\n" + "-"*60)
print("RESIDUAL PLOTS FOR BEST MODEL (training sample)")
print("-"*60)

# Determine best model by AIC
best_aic_model = df_comparison.loc[df_comparison['AIC'].idxmin()]

print(f"\n📈 Best model by AIC: {best_aic_model['Model']}")
print(f"   AIC = {best_aic_model['AIC']:.2f}")
print(f"   Features: {best_aic_model['Features']}")
print(f"   R²_train = {best_aic_model['R²_train']:.4f}")
print(f"   R²_test = {best_aic_model['R²_test']:.4f}")

# Find corresponding predictions on training sample
model_mapping = {
    'Full Ridge (all features)': y_train_pred_new,
    'Manual removal': y_train_pred_manual,
    'Stepwise selection': y_train_pred_step
}
y_train_pred_best = model_mapping[best_aic_model['Model']]

# Calculate residuals on TRAINING sample
residuals_train_best = y_train.values - y_train_pred_best

# Create plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Residual Diagnostics (training sample, n={n_train})\n{best_aic_model["Model"]}',
             fontsize=14, fontweight='bold')

# 1. Histogram of residuals with normal distribution curve
ax = axes[0, 0]
ax.hist(residuals_train_best, bins=20, edgecolor='black', alpha=0.7, density=True)
ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero')
# Overlay normal distribution curve
from scipy.stats import norm
x_range = np.linspace(residuals_train_best.min(), residuals_train_best.max(), 100)
ax.plot(x_range, norm.pdf(x_range, residuals_train_best.mean(), residuals_train_best.std()),
        'r-', linewidth=2, label='Normal distribution')
ax.set_xlabel('Residuals (billion RUB)')
ax.set_ylabel('Density')
ax.set_title('Residual Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Q-Q plot
ax = axes[0, 1]
from scipy import stats
stats.probplot(residuals_train_best, dist="norm", plot=ax)
ax.set_title('Q-Q plot (normality check)')
ax.grid(True, alpha=0.3)

# 3. Residuals vs predicted values
ax = axes[1, 0]
ax.scatter(y_train_pred_best, residuals_train_best, alpha=0.6, edgecolors='black', linewidth=0.5)
ax.axhline(y=0, color='red', linestyle='--', linewidth=2)
# Add smoothed trend line
from scipy.interpolate import make_interp_spline
try:
    sorted_idx = np.argsort(y_train_pred_best)
    x_sorted = y_train_pred_best[sorted_idx]
    y_sorted = residuals_train_best[sorted_idx]
    # Simple moving average for trend visualization
    window = 20
    y_smooth = np.convolve(y_sorted, np.ones(window)/window, mode='valid')
    x_smooth = x_sorted[window//2:window//2+len(y_smooth)]
    ax.plot(x_smooth, y_smooth, 'g-', linewidth=2, label=f'Moving average (window={window})')
except:
    pass
ax.set_xlabel('Predicted values (billion RUB)')
ax.set_ylabel('Residuals (billion RUB)')
ax.set_title('Residuals vs Predicted (homoscedasticity check)')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Autocorrelation function of residuals (ACF)
ax = axes[1, 1]
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(residuals_train_best, lags=min(20, n_train//4), ax=ax,
         title='Autocorrelation Function of Residuals (ACF)')
ax.set_xlabel('Lag')
ax.set_ylabel('Autocorrelation')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'residuals_diagnostics_{best_aic_model["Model"].replace(" ", "_")}.png',
            dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✅ Plots saved: 04_residuals_diagnostics_{best_aic_model['Model'].replace(' ', '_')}.png")

# -------------------------------------------------------------------
# 10.6. FINAL SUMMARY (corrected)
# -------------------------------------------------------------------

print("\n" + "="*60)
print("📌 FINAL SUMMARY ON MODEL COMPARISON")
print("="*60)

print("\n📊 FINAL SUMMARY TABLE:")
print("=" * 90)
print(f"{'Model':<30} {'Feat.':<8} {'R²_train':<10} {'R²_adj':<10} {'R²_test':<10} {'AIC':<12} {'BIC':<12}")
print("=" * 90)
for _, row in df_comparison.iterrows():
    print(f"{row['Model']:<30} {row['Features']:<8} {row['R²_train']:<10.4f} {row['R²_adj_train']:<10.4f} {row['R²_test']:<10.4f} {row['AIC']:<12.2f} {row['BIC']:<12.2f}")

# Determine best models by different criteria
best_r2_train = df_comparison.loc[df_comparison['R²_train'].idxmax()]
best_r2_test = df_comparison.loc[df_comparison['R²_test'].idxmax()]
best_aic = df_comparison.loc[df_comparison['AIC'].idxmin()]
best_bic = df_comparison.loc[df_comparison['BIC'].idxmin()]
best_adj_r2 = df_comparison.loc[df_comparison['R²_adj_train'].idxmax()]

print("\n" + "=" * 60)
print("🏆 BEST MODELS BY DIFFERENT CRITERIA:")
print("=" * 60)
print(f"📊 By R²_train (goodness of fit): {best_r2_train['Model']} ({best_r2_train['R²_train']:.4f})")
print(f"📊 By R²_adj_train (accounting for complexity): {best_adj_r2['Model']} ({best_adj_r2['R²_adj_train']:.4f})")
print(f"📊 By R²_test (predictive ability): {best_r2_test['Model']} ({best_r2_test['R²_test']:.4f})")
print(f"📊 By AIC (information criterion): {best_aic['Model']} ({best_aic['AIC']:.2f})")
print(f"📊 By BIC (with stronger penalty): {best_bic['Model']} ({best_bic['BIC']:.2f})")

print("\n" + "=" * 60)
print("📌 MODEL SELECTION RECOMMENDATION:")
print("=" * 60)

# Analyze criteria consensus
criteria_winners = [
    best_r2_train['Model'],
    best_adj_r2['Model'],
    best_r2_test['Model'],
    best_aic['Model'],
    best_bic['Model']
]

from collections import Counter
winner_counts = Counter(criteria_winners)
most_common_winner = winner_counts.most_common(1)[0]

print(f"\n📊 Criteria consensus:")
for model, count in winner_counts.items():
    print(f"   {model}: {count} criteria out of 5")

if most_common_winner[1] >= 3:
    print(f"\n✅ Majority of criteria (≥3) select model: {most_common_winner[0]}")
    recommended_model = most_common_winner[0]
else:
    print(f"\n⚠️ Criteria disagree")
    # When criteria disagree, prefer AIC (balance of quality/complexity)
    recommended_model = best_aic['Model']
    print(f"   Recommend {recommended_model} (by AIC - balance of quality/complexity)")

print(f"\n🎯 RECOMMENDED MODEL: {recommended_model}")

# Print detailed information about recommended model
rec_model_data = df_comparison[df_comparison['Model'] == recommended_model].iloc[0]
print(f"\n📋 Recommended model characteristics:")
print(f"   • Number of features: {rec_model_data['Features']}")
print(f"   • R² on training: {rec_model_data['R²_train']:.4f}")
print(f"   • Adjusted R²: {rec_model_data['R²_adj_train']:.4f}")
print(f"   • R² on test: {rec_model_data['R²_test']:.4f}")
print(f"   • AIC: {rec_model_data['AIC']:.2f}")
print(f"   • BIC: {rec_model_data['BIC']:.2f}")
print(f"   • RMSE on test: {rec_model_data['RMSE_test']:.2f} billion RUB")

# Comparison with full Ridge model
if recommended_model != 'Full Ridge (all features)':
    full_model_data = df_comparison[df_comparison['Model'] == 'Full Ridge (all features)'].iloc[0]
    reduction = full_model_data['Features'] - rec_model_data['Features']
    aic_diff = full_model_data['AIC'] - rec_model_data['AIC']
    print(f"\n📊 Compared to full model:")
    print(f"   • Feature reduction: {reduction} ({reduction/full_model_data['Features']*100:.1f}%)")
    print(f"   • AIC improvement: {aic_diff:.2f}")
    if rec_model_data['R²_test'] > full_model_data['R²_test']:
        print(f"   • Predictive ability improvement: +{rec_model_data['R²_test'] - full_model_data['R²_test']:.4f}")
    else:
        print(f"   • Predictive ability change: {rec_model_data['R²_test'] - full_model_data['R²_test']:.4f}")

print("\n" + "=" * 60)
print("📌 KEY FINDINGS:")
print("=" * 60)
print("""
1. All goodness-of-fit metrics (R²_train, R²_adj, AIC, BIC)
   are calculated on the TRAINING sample for correct model comparison.

2. Predictive ability (R²_test, RMSE, MAE) is assessed
   on the TEST sample (12 most recent observations).

3. Residual diagnostics performed on the TRAINING sample:
   - Normality: Shapiro-Wilk test
   - Homoscedasticity: Breusch-Pagan test
   - Autocorrelation: Breusch-Godfrey test (instead of DW)

4. F-test is excluded from analysis as it is incorrect
   for Ridge regression (biased estimates).

5. Breusch-Godfrey test is preferred over DW test because:
   - Correctly handles lagged variables
   - Allows testing different orders of autocorrelation
   - More powerful for small samples

6. For final model selection, a consensus of
   multiple criteria is used with AIC priority.
""")

print("\n✅ Removing insignificant features completed")

print("\n✅ Block 4 completed")